# ResNet-18 trained on synthetic defects, tested on real ones

Can a defect classifier be trained when real defect images are scarce? Here the **defect** class of the training set
consists only of images generated by the class-conditional diffusion model (`2_conditional_ddpm_defect_generation.ipynb`),
while the **good** class uses real crops. The test set contains **only real images** (good and defective).

* Pre-processing: crop of the 36-px stripe of interest, grayscale → 3 channels, resize to 224, ImageNet normalisation,
  horizontal-flip augmentation.
* Two transfer-learning variants of an ImageNet ResNet-18 (SGD + momentum, StepLR, 25 epochs, best-validation checkpoint):
  1. **full fine-tuning** of all layers;
  2. **frozen backbone**, only the final linear layer is trained.
* Validation is a 30 % split of the (synthetic-defect + real-good) training pool, so it does not measure the
  synthetic→real gap; the real test set does.

**Data:** proprietary line-scan crops and generated images (not included).


In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

# Training pool: real good crops + diffusion-generated defect crops
img_dir1 = f"{DATA_ROOT}/line_scan_crops/train/good"
img_dir2 = f"{DATA_ROOT}/synthetic_defects/ddpm/class_0"
# Alternatives that were tried for the synthetic defect class:
#   f"{DATA_ROOT}/synthetic_defects/wgan_gp"      (1_wgan_gp_synthetic_defects.ipynb)
# Test set: real images only
test_img_dir1 = f"{DATA_ROOT}/line_scan_crops/test/good"
test_img_dir2 = f"{DATA_ROOT}/line_scan_crops/test/bad"

In [ ]:
%pip install -q torch torchvision tqdm pandas matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
from PIL import Image
from tempfile import TemporaryDirectory

from torch.utils.data import DataLoader
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset
import pandas as pd
import os
from torchvision.io import read_image
from torch.utils.data import random_split
cudnn.benchmark = True
import torch
import numpy as np
import random

def set_seed(seed):
    """Sets the random seed for reproducibility."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set a constant seed at the beginning of your script
SEED = 42 # You can choose any integer
set_seed(SEED)

In [ ]:
import os
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset

class ResnetImageDataset(Dataset):
    def __init__(self, img_dir1, img_dir2, transform=None, target_transform=None):
        self.img_dir1 = img_dir1
        self.img_dir2 = img_dir2
        self.transform = transform
        self.target_transform = target_transform

        # Get all image files in each directory
        self.img_files1 = [f for f in os.listdir(img_dir1) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.img_files2 = [f for f in os.listdir(img_dir2) if f.endswith(('.jpg', '.jpeg', '.png'))]

        # Combine image files and labels
        self.img_files = self.img_files1 + self.img_files2
        self.labels = [0] * len(self.img_files1) + [1] * len(self.img_files2)

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        # Use a loop to handle potential errors and find a valid image
        while True:
            try:
                # Get image file and label
                img_file = self.img_files[idx]
                label = self.labels[idx]

                # Get image path based on label
                img_path = os.path.join(self.img_dir1 if label == 0 else self.img_dir2, img_file)

                # Read image using Pillow to get a PIL Image
                image = Image.open(img_path).convert('RGB') # Ensure image is in RGB format

                # Define crop boundaries
                left, upper = 0, 46  # Assuming you want to crop from the left edge and 46 pixels from the top
                right, lower = image.width, 82 # Assuming you want to crop to the right edge and 82 pixels from the top

                # Perform the crop
                image = image.crop((left, upper, right, lower))

                # Apply the rest of the transformations (excluding the Resize one)
                # Make sure your 'transform' Compose includes the Resize
                if self.transform:
                    image = self.transform(image)

                if self.target_transform:
                    label = self.target_transform(label)

                # If we reached here, the image was loaded, cropped, and transformed successfully
                return image, label

            except Exception as e:
                # If an error occurs, print a warning and try the next index
                print(f"Warning: Skipping corrupted or problematic image at index {idx} ({self.img_files[idx]}): {e}")
                # Increment index and loop again.
                idx = (idx + 1) % len(self.img_files)

# Define the transformations with the resize step
data_transforms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
        transforms.Resize(224), # Resize after cropping
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
        transforms.Resize(224), # Resize after cropping
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Create dataset instance with transformations
dataset = ResnetImageDataset(img_dir1, img_dir2)

In [ ]:
train_size = int(0.7 * len(dataset))  # 70% for training
val_size = len(dataset) - train_size  # 30% for validation

# Perform the split
train_dataset, val_dataset= random_split(dataset, [train_size, val_size])
test_dataset = ResnetImageDataset(test_img_dir1, test_img_dir2)
# Create DataLoaders
train_dl = DataLoader(train_dataset, batch_size=1)
val_dl = DataLoader(val_dataset, batch_size=1)  # DataLoader for validation set
test_dl = DataLoader(test_dataset, batch_size=1)
# 3. Create new datasets with transforms applied:
class TransformedDataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Apply transforms to the split datasets
train_dataset_transformed = TransformedDataset(train_dataset, data_transforms['train'])
val_dataset_transformed = TransformedDataset(val_dataset, data_transforms['val'])
test_dataset_transformed = TransformedDataset(test_dataset, data_transforms['val'])  # Using 'val' transforms for test as well

# 4. Create DataLoaders:
train_dl = DataLoader(train_dataset_transformed, batch_size=8, shuffle=True, num_workers=2)
val_dl = DataLoader(val_dataset_transformed, batch_size=8, shuffle=False, num_workers=2)
test_dl = DataLoader(test_dataset_transformed, batch_size=8, shuffle=False, num_workers=2)

dataloaders = {
    'train': train_dl,
    'val': val_dl,
    'test': test_dl  # Added test_dl
}

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated


# Get a batch of training data
inputs, classes = next(iter(dataloaders['train']))

# Make a grid from batch
out = torchvision.utils.make_grid(inputs)
class_names = ['good', 'defect']


imshow(out, title=[class_names[x] for x in classes])

In [ ]:
from tqdm import tqdm

def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()

    # Create a temporary directory to save training checkpoints
    with TemporaryDirectory() as tempdir:
        os.makedirs("outputs", exist_ok=True)
        best_model_params_path = "outputs/best_model_params.pt"

        torch.save(model.state_dict(), best_model_params_path)
        best_acc = 0.0

        for epoch in range(num_epochs): # Keep the outer loop without tqdm
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)

            # Each epoch has a training and validation phase
            for phase in ['train', 'val']:
                if phase == 'train':
                    model.train()  # Set model to training mode
                else:
                    model.eval()   # Set model to evaluate mode

                running_loss = 0.0
                running_corrects = 0

                # Iterate over data with a progress bar
                for inputs, labels in tqdm(dataloaders[phase], desc=f'{phase.capitalize()} Phase'):
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # zero the parameter gradients
                    optimizer.zero_grad()

                    # forward
                    # track history if only in train
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        # backward + optimize only if in training phase
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    # statistics
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                if phase == 'train':
                    scheduler.step()

                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects.double() / dataset_sizes[phase]

                print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                # deep copy the model
                if phase == 'val' and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), best_model_params_path)

            print()

        time_elapsed = time.time() - since
        print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
        print(f'Best val Acc: {best_acc:4f}')

        # load best model weights
        model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
    return model

In [ ]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['val']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title(f'predicted: {class_names[preds[j]]}')
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory

cudnn.benchmark = True
plt.ion()   # interactive mode

# Define device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # Check for GPU availability, otherwise use CPU

model_ft = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features
# Here the size of each output sample is set to 2.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
model_ft.fc = nn.Linear(num_ftrs, 2)

model_ft = model_ft.to(device) # Now 'device' is defined

criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)
dataset_sizes = {'train': len(train_dataset_transformed),
                 'val': len(val_dataset_transformed),
                 'test': len(test_dataset_transformed)}

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


In [ ]:
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler,
                       num_epochs=25)

Epoch 0/24
----------
train Loss: 0.2523 Acc: 0.8871
val Loss: 0.0109 Acc: 1.0000

Epoch 1/24
----------
train Loss: 0.1130 Acc: 0.9671
val Loss: 0.0233 Acc: 0.9967

Epoch 2/24
----------
train Loss: 0.1255 Acc: 0.9629
val Loss: 0.0288 Acc: 0.9967

Epoch 3/24
----------
train Loss: 0.0628 Acc: 0.9757
val Loss: 0.0033 Acc: 1.0000

Epoch 4/24
----------
train Loss: 0.0522 Acc: 0.9786
val Loss: 0.0123 Acc: 0.9967

Epoch 5/24
----------
train Loss: 0.0433 Acc: 0.9900
val Loss: 0.0001 Acc: 1.0000

Epoch 6/24
----------
train Loss: 0.0371 Acc: 0.9929
val Loss: 0.0004 Acc: 1.0000

Epoch 7/24
----------
train Loss: 0.0215 Acc: 0.9929
val Loss: 0.0003 Acc: 1.0000

Epoch 8/24
----------
train Loss: 0.0085 Acc: 0.9986
val Loss: 0.0002 Acc: 1.0000

Epoch 9/24
----------
train Loss: 0.0070 Acc: 0.9986
val Loss: 0.0003 Acc: 1.0000

Epoch 10/24
----------
train Loss: 0.0537 Acc: 0.9757
val Loss: 0.0005 Acc: 1.0000

Epoch 11/24
----------
train Loss: 0.0163 Acc: 0.9929
val Loss: 0.0003 Acc: 1.0000

Ep

In [ ]:
from tqdm import tqdm
import time # Import the time module

print("Evaluating on test set and benchmarking inference speed...")

model_ft.eval()  # Set model to evaluation mode

running_corrects = 0
total_images = 0
inference_start_time = time.time() # Start time measurement

# Wrap your DataLoader with tqdm
with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="Testing Progress"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model_ft(inputs)
        _, preds = torch.max(outputs, 1)

        running_corrects += torch.sum(preds == labels.data)
        total_images += inputs.size(0)

inference_end_time = time.time() # End time measurement
total_inference_time = inference_end_time - inference_start_time

test_acc = running_corrects.double() / total_images

print(f'Test Acc: {test_acc:.4f}')
print(f'Total Inference Time: {total_inference_time:.4f} seconds')
print(f'Average Inference Time per Image: {total_inference_time / total_images:.4f} seconds')

Evaluating on test set and benchmarking inference speed...
Test Acc: 0.8780
Total Inference Time: 5.5939 seconds
Average Inference Time per Image: 0.0056 seconds


In [ ]:
model_conv = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_conv.parameters():
    param.requires_grad = False

# Parameters of newly constructed modules have requires_grad=True by default
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Linear(num_ftrs, 2)

model_conv = model_conv.to(device)

criterion = nn.CrossEntropyLoss()

# Observe that only parameters of final layer are being optimized as
# opposed to before.
optimizer_conv = optim.SGD(model_conv.fc.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

In [ ]:
model_conv = train_model(model_conv, criterion, optimizer_conv,
                         exp_lr_scheduler, num_epochs=25)

Epoch 0/24
----------
train Loss: 0.3708 Acc: 0.8400
val Loss: 0.1002 Acc: 0.9767

Epoch 1/24
----------
train Loss: 0.2151 Acc: 0.9171
val Loss: 0.0586 Acc: 0.9967

Epoch 2/24
----------
train Loss: 0.1861 Acc: 0.9186
val Loss: 0.0383 Acc: 1.0000

Epoch 3/24
----------
train Loss: 0.1378 Acc: 0.9486
val Loss: 0.0236 Acc: 1.0000

Epoch 4/24
----------
train Loss: 0.1561 Acc: 0.9443
val Loss: 0.0413 Acc: 0.9867

Epoch 5/24
----------
train Loss: 0.1126 Acc: 0.9643
val Loss: 0.0217 Acc: 1.0000

Epoch 6/24
----------
train Loss: 0.1503 Acc: 0.9443
val Loss: 0.0156 Acc: 1.0000

Epoch 7/24
----------
train Loss: 0.1192 Acc: 0.9614
val Loss: 0.0156 Acc: 1.0000

Epoch 8/24
----------
train Loss: 0.1026 Acc: 0.9629
val Loss: 0.0179 Acc: 1.0000

Epoch 9/24
----------
train Loss: 0.0839 Acc: 0.9729
val Loss: 0.0168 Acc: 1.0000

Epoch 10/24
----------
train Loss: 0.0742 Acc: 0.9857
val Loss: 0.0148 Acc: 1.0000

Epoch 11/24
----------
train Loss: 0.0738 Acc: 0.9814
val Loss: 0.0167 Acc: 1.0000

Ep

In [ ]:
from tqdm import tqdm
import time # Import the time module

print("Evaluating on test set and benchmarking inference speed...")

model_conv.eval()  # Set model to evaluation mode

running_corrects = 0
total_images = 0
inference_start_time = time.time() # Start time measurement

# Wrap your DataLoader with tqdm
with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="Testing Progress"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model_conv(inputs)
        _, preds = torch.max(outputs, 1)

        running_corrects += torch.sum(preds == labels.data)
        total_images += inputs.size(0)

inference_end_time = time.time() # End time measurement
total_inference_time = inference_end_time - inference_start_time

test_acc = running_corrects.double() / total_images

print(f'Test Acc: {test_acc:.4f}')
print(f'Total Inference Time: {total_inference_time:.4f} seconds')
print(f'Average Inference Time per Image: {total_inference_time / total_images:.4f} seconds')

Evaluating on test set and benchmarking inference speed...
Test Acc: 0.8140
Total Inference Time: 6.8603 seconds
Average Inference Time per Image: 0.0069 seconds
